# Smart Farming Agent — EDA & Training Notebook
This notebook walks through data exploration, model training, and evaluation.

In [ ]:
import sys, os
sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config import DISEASE_CLASSES, OPTIMAL_RANGES

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 5)
print('Environment ready ✓')

## 1. Soil Data EDA

In [ ]:
from src.models.soil_model import SoilHealthAnalyzer
analyzer = SoilHealthAnalyzer()
df = analyzer.generate_synthetic_data(n_samples=3000)
print(df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
features = ['N', 'P', 'K', 'pH', 'moisture', 'EC', 'organic_matter', 'temperature']
for ax, feat in zip(axes.flat, features):
    df[feat].hist(ax=ax, bins=40, color='#2ECC71', edgecolor='white', alpha=0.85)
    ax.set_title(feat, fontsize=12)
    ax.set_xlabel('')
plt.suptitle('Soil Parameter Distributions', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Crop distribution
df['crop'].value_counts().plot(kind='barh', color='#3498DB')
plt.title('Crop Class Distribution')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = df[features].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5)
plt.title('Soil Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 2. Train Soil Model

In [ ]:
from src.models.soil_model import SoilHealthAnalyzer
analyzer = SoilHealthAnalyzer.__new__(SoilHealthAnalyzer)
analyzer.model = None
analyzer.loaded = False
df = analyzer.generate_synthetic_data(5000)
analyzer.train(df)
print('Training complete!')

## 3. Soil Analysis Demo

In [ ]:
from src.utils.data_utils import dummy_soil_params
params = dummy_soil_params()
health = analyzer.analyze_health(params)
crop_rec = analyzer.predict_crop(params)

print(f"Overall Soil Score: {health['overall_score']}/100 — {health['grade']}")
print(f"Recommended Crop:  {crop_rec['recommended_crop']} ({crop_rec['confidence']:.1%})")
print(f"Issues:            {health['issues'] or 'None'}")
print("\nRecommendations:")
for r in health['recommendations']:
    print(' •', r)

## 4. Weather Forecasting

In [ ]:
from src.models.weather_model import WeatherForecaster
wf = WeatherForecaster()
fc = wf.predict_7day()
print(fc.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.fill_between(fc['date'], fc['temperature_min'], fc['temperature_max'],
                  alpha=0.3, color='#E74C3C', label='Temp range')
ax1.plot(fc['date'], fc['temperature'], 'o-', color='#E74C3C', linewidth=2, label='Mean temp')
ax1.set_ylabel('Temperature (°C)')
ax1.legend()
ax1.set_title('7-Day Weather Forecast')

ax2.bar(fc['date'], fc['rainfall'], color='#3498DB', alpha=0.8, label='Rainfall')
ax2.set_ylabel('Rainfall (mm)')
ax2.set_xlabel('Date')
ax2.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. CNN Disease Detection (mock)

In [ ]:
from src.models.cnn_disease_model import CropDiseaseDetector
detector = CropDiseaseDetector()

# Simulate 10 predictions
results = []
for i in range(10):
    img = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
    r = detector.predict(img)
    results.append({'prediction': r['top_prediction'], 'confidence': r['confidence'], 'healthy': r['is_healthy']})

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))
print(f"\nHealthy rate: {res_df['healthy'].mean():.0%}")

## 6. Full Agent Run

In [ ]:
import json
from src.agents.farming_agent import SmartFarmingAgent
agent = SmartFarmingAgent()

dummy_img = np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
report = agent.analyze(
    image_array=dummy_img,
    soil_params=dummy_soil_params(),
    crop_name='Tomato'
)

print('Alert Level:', report['alert_level'])
print('Action Plan:')
for step in report['action_plan']:
    print(f"  Day {step.get('day','?')} [{step.get('priority','?')}] {step.get('category','')}: {step.get('action','')[:80]}")